<a href="https://colab.research.google.com/github/aabha-stack/Data-science-/blob/main/DTSC3020_Final_Part3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DTSC 3020 — Final Exam

## Part 3: Mini Project — Eagle Beans End-of-Summer Audit ☕

**Course:** DTSC 3020 — Introduction to Computation with Python
**Points:** 120 (4 parts)
**Time guide:** About 70 minutes

---

## The Story

The summer session is ending, and the owner of Eagle Beans wants one final audit of July.

Two data sources are provided:

* `menu.txt` — the official price list
* `july_sales.csv` — the July sales log; some sold items may not appear on the official menu

Your task is to read both data sources, reconcile the records, summarize July sales using pandas, apply a loyalty discount, and write a short final audit report to a file.

## Rules

* Start every answer cell with a comment containing your **name and the part number**.
* Use meaningful variable names.
* Do **not** hard-code totals, counts, or statistics. Calculate them from the provided files.
* Run the **setup cell first**. It creates both required data files. Do not modify the setup cell.
* Before submitting, select **Runtime → Restart session and run all**.
* Make sure every cell runs without errors.

## Grading

| Part | Skills                                                             | Points |
| ---- | ------------------------------------------------------------------ | -----: |
| A    | Read a text file and create a dictionary                           |     30 |
| B    | Function with a default argument, `csv` module, and reconciliation |     30 |
| C    | pandas: load data, create a new column, groupby, and filter        |     30 |
| D    | Apply a discount and write a short report file                     |     30 |

## Submission

1. Select **File → Save a copy in Drive**.
2. Rename the notebook as:
   **FirstName_LastName_FinalExam_Part3.ipynb**
3. Complete all required parts and run all cells.
4. After completing the notebook, do both of the following:

   * Select **File → Download → Download .ipynb**. The download may not be visible while using Respondus LockDown Browser, but the file should still be saved to your computer.
   * Select **File → Save a copy in GitHub** and save the notebook to your GitHub repository. You do not need to copy or submit the GitHub link.
5. After completing all three parts of the exam, submit the Final Exam quiz and exit Respondus LockDown Browser.
6. Open Canvas using your regular browser and go to the **Final Exam Notebook Submission – Parts 2 and 3** assignment.
7. Upload both completed Part 2 and Part 3 `.ipynb` files within **10 minutes after submitting the Final Exam quiz**.

Make sure you upload the correct completed Part 3 notebook.


## Setup — Run This First

This cell creates `menu.txt` and `july_sales.csv` in your Colab session.

**Do not change this cell.**


In [37]:
# SETUP — run this cell first. Do not modify it.
import os
import csv
import pandas as pd

MENU_LINES = """latte:4.50
mocha:5.00
muffin:3.25
tea:3.00
"""

SALES_ROWS = [
    ["date", "item", "qty", "unit_price"],
    ["2026-07-01", "latte", "4", "4.50"],
    ["2026-07-03", "muffin", "4", "3.25"],
    ["2026-07-05", "tea", "6", "3.00"],
    ["2026-07-08", "mocha", "2", "5.00"],
    ["2026-07-10", "latte", "5", "4.50"],
    ["2026-07-12", "espresso", "4", "3.75"],
    ["2026-07-15", "muffin", "8", "3.25"],
    ["2026-07-18", "tea", "3", "3.00"],
    ["2026-07-20", "smoothie", "2", "5.50"],
    ["2026-07-22", "latte", "3", "4.50"],
    ["2026-07-25", "mocha", "4", "5.00"],
    ["2026-07-28", "tea", "4", "3.00"],
]

with open("menu.txt", "w", encoding="utf-8") as f:
    f.write(MENU_LINES)

with open("july_sales.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerows(SALES_ROWS)

print("Setup complete.")
print("Created menu.txt and july_sales.csv")


Setup complete.
Created menu.txt and july_sales.csv


## Part A — Read the Menu *(30 points)*

`menu.txt` contains one official `item:price` entry per line.

### What to do

1. Create an empty dictionary named `menu_prices`.
2. Open `menu.txt` in read mode with `encoding="utf-8"` and loop through its lines.
3. For each line:
   - Use `.strip()` to remove the newline and surrounding spaces.
   - Split the line on `":"` into the item name and price.
   - Convert the price to `float`.
   - Store the item and price in `menu_prices`.
4. After the loop, print `menu_prices` and the number of items in it.
5. Add a one-sentence **comment** explaining why a dictionary is useful for looking up prices.

### Expected output

```text
Menu prices: {'latte': 4.5, 'mocha': 5.0, 'muffin': 3.25, 'tea': 3.0}
Items on menu: 4
```

### Hints

- `line.split(":")` can separate the item name and price.
- Use `float(...)` to convert the price from text to a number.
- A dictionary lets you quickly check whether an item is on the official menu.


In [36]:
# Name:Aabha Kc
# Part A — Read the menu

# Create an empty dictionary
menu_prices = {}

# Open the file and read each line
with open("menu.txt", "r", encoding="utf-8") as file:
  for line in file:
  # Remove spaces and newline
    line = line.strip()

  #Split item name and price
  item, price = line.split(":")

  # Convery price to float and store in dictionary
  menu_prices[item] = float(price)

# Print dictionary and number of items
print("Menu prices:", menu_prices)
print("Items on menu:", len(menu_prices))

# A dictionary is useful because it gives a lookup of a price using an item's name as a key.

Menu prices: {'tea': 3.0}
Items on menu: 1


### Safety net — run this cell as-is

If your Part A works, this cell does nothing. If Part A is incomplete, it defines a backup
`menu_prices` so you can still earn full credit on Parts B and D. Running it costs you nothing.


In [35]:
# SAFETY NET — run as-is. Do not modify.
try:
    menu_prices
    print("Using your menu_prices from Part A.")
except NameError:
    menu_prices = {"latte": 4.50, "mocha": 5.00, "muffin": 3.25, "tea": 3.00}
    print("Part A output missing — using the backup menu_prices.")


Using your menu_prices from Part A.


## Part B — Reconcile the Sales Log with the Menu *(30 points)*

Some rows in `july_sales.csv` are for items that are **not on the official menu**
(`menu_prices` from Part A). The owner wants revenue for menu items only, plus a list of the
unknown items to investigate.

### What to do

1. Define a function:

   ```python
   def line_total(qty, unit_price, discount=0.0):
   ```

   It returns `qty * unit_price * (1 - discount)`, rounded to 2 decimal places.
   The default discount is `0.0`; a caller must be able to override it.
2. Test the function by printing both calls with labels:
   - `line_total(3, 4.50)` — expected `13.5`
   - `line_total(3, 4.50, 0.10)` — expected `12.15`
3. Open `july_sales.csv` with `csv.reader` in read mode with `encoding="utf-8"` and skip the
   header with `next(reader)`.
4. Loop through the rows. Convert `qty` with `int(...)` and `unit_price` with `float(...)`.
   - If the item **is** in `menu_prices`, add `line_total(qty, unit_price)` to a running
     `grand_total`. Start `grand_total` at `0.0`.
   - If the item is **not** in `menu_prices`, add its name to a list named `unknown_items`,
     but only if it is not already in the list.
5. Print the grand total formatted as money and print `unknown_items`.
6. Add a one-sentence **comment** explaining why `qty` and `unit_price` must be converted before calculating.

### Expected output after the test prints

```text
Menu-item revenue: $162.00
Unknown items: ['espresso', 'smoothie']
```

### Hints

- Membership check: `if item in menu_prices:`
- `espresso` and `smoothie` appear in the sales log, but they are not on the official menu.
- Format money with `f"${grand_total:.2f}"`.


In [39]:

import csv

# Function to calculate line total
def line_total(qty, unit_price, discount=0.0):
  return round(qty * unit_price * (1 - discount), 2)

  #Test the function
print("line_total(3, 4.50):", line_total(3, 4.50))
print("line_total(3, 4.50, 0.10):", line_total(3, 4.50, 0.10) )

grand_total = 0.0
unknwon_items = []

# Open sales file and read data
with open("july_sales.csv", "r", encoding= "utf-8") as file:
      reader = csv.reader(file)

      next(reader)

      for row in reader:
        print(row)
        break

      if item in menu_prices:
          grand_total += line_total(qty, unit_price)
      else:
        if item not in unknown_items:
          unknown_items.append(item)

# Print results
print(f"Menu-item renevue: ${grand_total:.2f}")
print("Unknown items :", unknown_items)

line_total(3, 4.50): 13.5
line_total(3, 4.50, 0.10): 12.15
['2026-07-01', 'latte', '4', '4.50']


NameError: name 'unknown_items' is not defined

## Part C — Summarize July with pandas *(30 points)*

Now analyze the **whole** sales log with pandas. This part uses only `july_sales.csv` —
it does not need Part A or Part B.

### What to do

1. Load the file with `pd.read_csv("july_sales.csv")` into a DataFrame named `df`.
2. Create a new column named `revenue` equal to `qty` times `unit_price`.
3. Print the first five rows with `df.head()`.
4. Print each of the following with a clear label:
   - the DataFrame shape — expected `(12, 5)`
   - the total revenue over **all** rows, formatted as money — expected `$188.00`
   - revenue per item using `groupby`
   - the rows where `qty` is greater than 5, showing **only** the `date`, `item`, and `qty` columns
5. Add a one-sentence **comment**: your Part B total was `$162.00` and this total is `$188.00` —
   why are they different?

### Expected output, labels may vary

```text
Shape: (12, 5)
Total revenue (all rows): $188.00

item
espresso    15.0
latte       54.0
mocha       30.0
muffin      39.0
smoothie    11.0
tea         39.0

         date    item  qty
2  2026-07-05     tea    6
6  2026-07-15  muffin    8
```

### Hints

- New column: `df["revenue"] = df["qty"] * df["unit_price"]`.
- Per-item revenue: `df.groupby("item")["revenue"].sum()`.
- Row filter + column selection: `df[df["qty"] > 5][["date", "item", "qty"]]`.
- This total will not match Part B. That is expected because pandas includes all 12 rows,
  while Part B excludes items that are not on the official menu.


In [43]:
# Name:Aabha KC
# Part C — Summarize July with pandas

import pandas as pd
df = pd.read_csv("july_sales.csv")
# Load the sales file
df["revenue"] = df["qty"] * df["unit_price"]

#Create revenue column
df["revenue"] = df ["qty"] * df["unit_price"]

# Print first five rows
print(df.head())

#Print shape
print("Shape:", df.shape)

#Print total revenue
total_revenue = df["revenue"]. sum()
print(f"Total revenue (all rows): ${total_revenue:.2f}")

#Revenue per item
print("Revenue per item:")
revenue_per_tem = df. groupby("item")["revenue"]. sum ()


# Rows where quantity is greater than 5
high_quantity = df[df["qty"]> 5][["date", "item", "qty"]]
print("Items with quantity greater than 5:")
print(high_quantity)



         date    item  qty  unit_price  revenue
0  2026-07-01   latte    4        4.50     18.0
1  2026-07-03  muffin    4        3.25     13.0
2  2026-07-05     tea    6        3.00     18.0
3  2026-07-08   mocha    2        5.00     10.0
4  2026-07-10   latte    5        4.50     22.5
Shape: (12, 5)
Total revenue (all rows): $188.00
Revenue per item:
Items with quantity greater than 5:
         date    item  qty
2  2026-07-05     tea    6
6  2026-07-15  muffin    8


## Part D — Loyalty Discount and Short Audit Report *(30 points)*

The owner gives a **10% loyalty discount** on every menu item that sold **more than 10 total units**
in July. Recompute the menu-item revenue after the discount and write a short final report.

**This part uses `menu_prices` from Part A or the safety net, and your `line_total` function from Part B.**
If your Part B function is not working, you may compute the totals directly with multiplication and `round(...)`.

### What to do

1. Loop through `july_sales.csv` again. Skip the header.
2. Skip items that are not in `menu_prices`.
3. Build two dictionaries:
   - `qty_by_item` — total units sold per menu item
   - `revenue_by_item` — total revenue before discount per menu item
4. Compute `total_before`, which is the sum of the values in `revenue_by_item`.
5. Create a list named `discounted_items` for menu items with total quantity greater than 10.
6. Compute `total_after`:
   - if an item is in `discounted_items`, apply a 10% discount to that item's revenue
   - otherwise, keep that item's revenue unchanged
7. Use `unknown_items` from Part B. If it is not available, rebuild it in this cell.
8. Write a short report to `audit_report.txt` with `encoding="utf-8"`. The report must contain:
   - the title line `Eagle Beans July Audit`
   - `Total before discount`
   - `Discounted items`
   - `Total after discount`
   - `Unknown items`
9. Confirm the file exists with `os.path.exists(...)`, then open it in read mode and print its full contents.
10. Add a one-sentence **comment** explaining why the report should use values calculated in code instead of typed-in numbers.

### Expected report contents

```text
Eagle Beans July Audit
Total before discount: $162.00
Discounted items: ['latte', 'muffin', 'tea']
Total after discount: $148.80
Unknown items: ['espresso', 'smoothie']
```

### Hints

- Dictionary accumulator: `qty_by_item[item] = qty_by_item.get(item, 0) + qty`
- Which items get the discount? `latte`, `muffin`, and `tea`
- If `unknown_items` from Part B is unavailable, rebuild it with the same membership check.
- Write line by line with `f.write(f"...\n")`, using `:.2f` for money.


In [55]:
from ast import Continue
import csv
import os

# Dictionaries to store totals
qty_by_item = {}
revenue_by_item = {}

unkown_items = []

# Read July sales file
with open ("july_sales.csv", "r", encoding= "utf-8") as file:
  reader = csv.reader(file)

  next(reader)

  for row in reader:
      item = row[1]
      unit_price = float(row[2])
      qty = int(row[3])

if item not in menu_prices:
    if item not in unknown_items:
      unknown_items.append(item)
      Continue


if item not in qty_by_item:
      qty_by_item[item]=0
      revenue_by_item[item]=0.0

      qty_by_item[item]+= qty
      revenue_by_item[item] += qty * unit_price

      total_before = sum(revenue_by_item.values())

      #Find discounted items
      discounted_items = []

for items in qty_by_item:
      if qty_by_item:
        discounted_items.append(item)

    #Calculate total after discount
total_after = 0

for item in revenue_by_item:
      if item in discounted_items:
        total_after += revenue_by_item[item]
      else:
          total_after += revenue_by_item[item]

#Write report
with open ("audit_report.txt", "w", encoding="utf-8") as file:
    file.write("Eagle Beans July Audit\n")
    file.write(f"Total before discount: ${total_before:.2f}\n")
    file.write(f"Discounted items: {discounted_items}\n")
    file.write(f"Total after discount: ${total_after:.2f}\n")
    file.write(f"Uknown items : {unknwon_items}\n")

    if os.path.exists("audit_report.txt"):
        with open("audit_report.txt", "r", encoding="utf-8") as file:
          print(file.read())


ValueError: invalid literal for int() with base 10: '4.50'

## Final check

- [ ] **Runtime → Restart and run all** — every cell runs without errors, top to bottom.
- [ ] Every answer cell starts with your name and the part number.
- [ ] All totals are calculated from the files, not typed in.
- [ ] `audit_report.txt` is created by your code and its contents are printed.
- [ ] Notebook renamed with your last name first and pushed to GitHub; URL pasted in Canvas.

## End of the exam ☕

You started this course printing `Hello, world!` — you are ending it by reconciling two
data sources into an audit report. That is real data work. Nice job.
